# Taxon-wise positional k-mer analysis

This notebook performs a descriptive, taxon-wise analysis of overlapping
2-mers and 3-mers in mutually exclusive animal groups plus Angiosperms.

Key design decisions:

- Organisms are identified by their miRBase prefixes using the supplied,
  reviewed taxonomic manifest.
- Exact mature sequences are deduplicated **within each organism**.
  Identical sequences present in different organisms remain represented
  in every organism in which they occur.
- The primary taxon heatmap is the equally weighted mean of organism-level
  positional frequencies.
- Pooled, sequence-weighted frequencies and counts are also exported as a
  sensitivity analysis.
- Comparative heatmaps display start positions 0-19 for both k=2 and k=3.
  Full tail positions remain available in the exported tables.
- These are descriptive frequencies. The words `enriched` and `depleted`
  should be reserved for the later null-model analysis.


## Selected groups

The animal groups are mutually exclusive:

1. Fishes (non-tetrapod vertebrates)
2. Amphibians
3. Non-avian reptiles
4. Aves
5. Human
6. Non-human primates
7. Other mammals
8. Insects
9. Nematodes

Angiosperms comprise eudicots, monocots, and the basal angiosperm entry.
Other arthropods and residual metazoans are not pooled into artificial
comparison groups.


## Dependencies

Python 3.10 or later with `pandas`, `numpy`, `matplotlib`, and `seaborn`.

```python
# %pip install pandas numpy matplotlib seaborn
```


In [ ]:
from __future__ import annotations

import hashlib
import itertools
import json
import platform
import re
import sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", mpl.__version__)
print("seaborn:", sns.__version__)


## Configuration


In [ ]:
FASTA_PATH = Path(r"C:\Users\GCV\Downloads\mature.fa")
GROUP_MANIFEST_PATH = Path(
    r"C:\Users\GCV\Documents\Codex\2026-07-25\ca\outputs"
    r"\selected_taxon_groups.csv"
)
OUTPUT_DIR = Path.cwd() / "taxonwise_kmer_outputs"

EXPECTED_FASTA_SHA256 = (
    "3c521fc9bea3c7993e71cf188b11caf73e2d40ac83454e32876455317cc6d342"
)
EXPECTED_RAW_RECORDS = 48_885
EXPECTED_SELECTED_ORGANISMS = 185
EXPECTED_SELECTED_RAW_RECORDS = 43_564
EXPECTED_SELECTED_UNIQUE_WITHIN_ORGANISM = 40_854

EXPECTED_GROUP_SIZES = {
    "Fishes (non-tetrapod vertebrates)": 17,
    "Amphibians": 2,
    "Non-avian reptiles": 5,
    "Aves": 4,
    "Human": 1,
    "Non-human primates": 19,
    "Other mammals": 20,
    "Insects": 31,
    "Nematodes": 11,
    "Angiosperms": 75,
}

VALID_BASES = frozenset("ACGU")
BASE_ORDER = ("A", "C", "G", "U")
K_VALUES = (2, 3)
DISPLAY_POSITIONS = list(range(0, 20))

# Primary figures are species/organism-balanced.
PRIMARY_FREQUENCY_FIELD = "organism_balanced_mean_frequency_percent"
EXPORT_POOLED_HEATMAPS = False

COLOR_MAP = "RdPu"
FIGURE_BACKGROUND = "#fffafa"
GRID_COLOR = "#eadfdf"
FONT_FAMILY = "Arial"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Input FASTA:", FASTA_PATH.resolve())
print("Group manifest:", GROUP_MANIFEST_PATH.resolve())
print("Outputs:", OUTPUT_DIR.resolve())


## Validate the grouping manifest


In [ ]:
group_manifest = pd.read_csv(
    GROUP_MANIFEST_PATH,
    dtype={"organism_prefix": "string", "ncbi_taxid": "string"},
)

required_manifest_columns = {
    "group_order",
    "group_code",
    "analysis_group",
    "organism_prefix",
    "mirbase_organism_name",
    "raw_mature_records",
    "unique_sequences_within_organism",
}
missing_columns = required_manifest_columns - set(group_manifest.columns)
if missing_columns:
    raise ValueError(
        f"The group manifest is missing columns: {sorted(missing_columns)}"
    )

if group_manifest["organism_prefix"].duplicated().any():
    duplicates = group_manifest.loc[
        group_manifest["organism_prefix"].duplicated(keep=False),
        ["organism_prefix", "analysis_group"],
    ]
    raise ValueError(
        "An organism prefix was assigned to multiple groups:\n"
        f"{duplicates.to_string(index=False)}"
    )

observed_group_sizes = (
    group_manifest.groupby("analysis_group")["organism_prefix"]
    .nunique()
    .to_dict()
)
assert observed_group_sizes == EXPECTED_GROUP_SIZES, (
    "The selected group membership differs from the reviewed design.\n"
    f"Observed: {observed_group_sizes}"
)
assert len(group_manifest) == EXPECTED_SELECTED_ORGANISMS

group_order = (
    group_manifest[
        ["group_order", "group_code", "analysis_group"]
    ]
    .drop_duplicates()
    .sort_values("group_order")
    .reset_index(drop=True)
)
group_names = group_order["analysis_group"].tolist()
group_codes = dict(
    zip(group_order["analysis_group"], group_order["group_code"])
)

display(
    group_manifest.groupby(
        ["group_order", "group_code", "analysis_group"],
        as_index=False,
    ).agg(
        organism_entries=("organism_prefix", "nunique"),
        manifest_raw_records=("raw_mature_records", "sum"),
        manifest_unique_within_organism=(
            "unique_sequences_within_organism",
            "sum",
        ),
    ).sort_values("group_order")
)


## FASTA parsing, validation, and within-organism deduplication


In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def parse_fasta(path: Path) -> list[dict[str, str]]:
    records: list[dict[str, str]] = []
    header: str | None = None
    sequence_parts: list[str] = []

    with path.open("r", encoding="utf-8") as handle:
        for line_number, raw_line in enumerate(handle, start=1):
            line = raw_line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append(
                        {
                            "header": header,
                            "sequence": "".join(sequence_parts).upper(),
                        }
                    )
                header = line[1:].strip()
                sequence_parts = []
            else:
                if header is None:
                    raise ValueError(
                        f"Sequence before a header at line {line_number}."
                    )
                sequence_parts.append(line)

    if header is not None:
        records.append(
            {
                "header": header,
                "sequence": "".join(sequence_parts).upper(),
            }
        )
    if not records:
        raise ValueError(f"No FASTA records found in {path}.")
    return records


def organism_prefix_from_header(header: str) -> str:
    mature_id = header.split()[0]
    return mature_id.split("-", 1)[0]


def validate_records(records: list[dict[str, str]]) -> None:
    invalid = []
    for record in records:
        sequence = record["sequence"]
        invalid_characters = sorted(set(sequence) - VALID_BASES)
        if not sequence or invalid_characters:
            invalid.append(
                {
                    "header": record["header"],
                    "invalid_characters": "".join(invalid_characters),
                }
            )
    if invalid:
        raise ValueError(
            f"{len(invalid)} records failed A/C/G/U validation. "
            f"First examples: {invalid[:10]}"
        )


In [ ]:
if not FASTA_PATH.is_file():
    raise FileNotFoundError(FASTA_PATH)
if not GROUP_MANIFEST_PATH.is_file():
    raise FileNotFoundError(GROUP_MANIFEST_PATH)

fasta_sha256 = sha256_file(FASTA_PATH)
assert fasta_sha256 == EXPECTED_FASTA_SHA256

records = parse_fasta(FASTA_PATH)
validate_records(records)
assert len(records) == EXPECTED_RAW_RECORDS

selected_prefixes = set(group_manifest["organism_prefix"])
prefix_to_group = group_manifest.set_index("organism_prefix")[
    "analysis_group"
].to_dict()
prefix_to_name = group_manifest.set_index("organism_prefix")[
    "mirbase_organism_name"
].to_dict()

selected_raw_records = [
    record
    for record in records
    if organism_prefix_from_header(record["header"]) in selected_prefixes
]
assert len(selected_raw_records) == EXPECTED_SELECTED_RAW_RECORDS

sequences_by_prefix: defaultdict[str, set[str]] = defaultdict(set)
raw_count_by_prefix: Counter[str] = Counter()
for record in selected_raw_records:
    prefix = organism_prefix_from_header(record["header"])
    raw_count_by_prefix[prefix] += 1
    sequences_by_prefix[prefix].add(record["sequence"])

missing_selected_prefixes = selected_prefixes - set(sequences_by_prefix)
if missing_selected_prefixes:
    raise AssertionError(
        f"Selected prefixes absent from FASTA: {missing_selected_prefixes}"
    )

observed_unique_total = sum(
    len(sequences) for sequences in sequences_by_prefix.values()
)
assert observed_unique_total == EXPECTED_SELECTED_UNIQUE_WITHIN_ORGANISM

observed_membership = group_manifest.copy()
observed_membership["observed_raw_records"] = observed_membership[
    "organism_prefix"
].map(raw_count_by_prefix)
observed_membership["observed_unique_sequences"] = observed_membership[
    "organism_prefix"
].map(lambda prefix: len(sequences_by_prefix[prefix]))
observed_membership["duplicates_removed_within_organism"] = (
    observed_membership["observed_raw_records"]
    - observed_membership["observed_unique_sequences"]
)

if not (
    observed_membership["observed_raw_records"].astype(int)
    == observed_membership["raw_mature_records"].astype(int)
).all():
    raise AssertionError(
        "Observed FASTA counts differ from the taxonomy manifest."
    )
if not (
    observed_membership["observed_unique_sequences"].astype(int)
    == observed_membership[
        "unique_sequences_within_organism"
    ].astype(int)
).all():
    raise AssertionError(
        "Within-organism unique counts differ from the taxonomy manifest."
    )

observed_membership.to_csv(
    OUTPUT_DIR / "selected_taxon_membership_observed_counts.csv",
    index=False,
)
display(
    observed_membership.groupby(
        ["group_order", "group_code", "analysis_group"],
        as_index=False,
    ).agg(
        organisms=("organism_prefix", "nunique"),
        raw_records=("observed_raw_records", "sum"),
        unique_within_organism=("observed_unique_sequences", "sum"),
        within_organism_duplicates=(
            "duplicates_removed_within_organism",
            "sum",
        ),
    ).sort_values("group_order")
)


## Organism-level overlapping k-mer tables


In [ ]:
def ordered_kmers(k: int) -> list[str]:
    return [
        "".join(characters)
        for characters in itertools.product(BASE_ORDER, repeat=k)
    ]


def organism_kmer_long(
    prefix: str,
    sequences: set[str],
    k: int,
) -> pd.DataFrame:
    motifs = ordered_kmers(k)
    maximum_position = max(len(sequence) - k for sequence in sequences)
    motif_to_index = {motif: index for index, motif in enumerate(motifs)}

    counts = np.zeros(
        (maximum_position + 1, len(motifs)),
        dtype=np.int64,
    )
    eligible = np.zeros(maximum_position + 1, dtype=np.int64)

    for sequence in sorted(sequences):
        for position in range(len(sequence) - k + 1):
            motif = sequence[position : position + k]
            counts[position, motif_to_index[motif]] += 1
            eligible[position] += 1

    if not np.array_equal(counts.sum(axis=1), eligible):
        raise AssertionError(
            f"Counts do not equal denominators for {prefix}, k={k}."
        )

    frequencies = counts / eligible[:, None] * 100.0
    if not np.allclose(frequencies.sum(axis=1), 100.0, atol=1e-10):
        raise AssertionError(
            f"Frequency rows do not sum to 100 for {prefix}, k={k}."
        )

    frame = pd.DataFrame(
        {
            "organism_prefix": np.repeat(
                prefix, (maximum_position + 1) * len(motifs)
            ),
            "mirbase_organism_name": np.repeat(
                prefix_to_name[prefix],
                (maximum_position + 1) * len(motifs),
            ),
            "analysis_group": np.repeat(
                prefix_to_group[prefix],
                (maximum_position + 1) * len(motifs),
            ),
            "k": np.repeat(
                k, (maximum_position + 1) * len(motifs)
            ),
            "start_position_0based": np.repeat(
                np.arange(maximum_position + 1), len(motifs)
            ),
            "kmer": motifs * (maximum_position + 1),
            "count": counts.reshape(-1),
            "eligible_unique_sequences": np.repeat(
                eligible, len(motifs)
            ),
            "frequency_percent": frequencies.reshape(-1),
        }
    )
    return frame


organism_level_results: dict[int, pd.DataFrame] = {}
for k in K_VALUES:
    organism_level = pd.concat(
        [
            organism_kmer_long(prefix, sequences_by_prefix[prefix], k)
            for prefix in sorted(selected_prefixes)
        ],
        ignore_index=True,
    )
    organism_level_results[k] = organism_level
    organism_level.to_csv(
        OUTPUT_DIR
        / f"organism_level_{k}mer_positional_tidy.csv.gz",
        index=False,
        compression="gzip",
        float_format="%.10f",
    )
    print(
        f"k={k}: {len(organism_level):,} organism-position-motif rows"
    )


## Taxon aggregation: primary organism-balanced and pooled sensitivity


In [ ]:
def aggregate_taxon_results(
    organism_level: pd.DataFrame,
    k: int,
) -> pd.DataFrame:
    grouped = organism_level.groupby(
        ["analysis_group", "start_position_0based", "kmer"],
        sort=False,
        observed=True,
    )
    aggregate = grouped.agg(
        pooled_count=("count", "sum"),
        pooled_eligible_unique_sequences=(
            "eligible_unique_sequences",
            "sum",
        ),
        contributing_organisms=("organism_prefix", "nunique"),
        organism_balanced_mean_frequency_percent=(
            "frequency_percent",
            "mean",
        ),
        organism_balanced_sd_frequency_percent=(
            "frequency_percent",
            "std",
        ),
    ).reset_index()

    aggregate["organism_balanced_se_frequency_percent"] = (
        aggregate["organism_balanced_sd_frequency_percent"]
        / np.sqrt(aggregate["contributing_organisms"])
    )
    aggregate["pooled_frequency_percent"] = (
        aggregate["pooled_count"]
        / aggregate["pooled_eligible_unique_sequences"]
        * 100.0
    )
    aggregate["k"] = k
    aggregate["group_order"] = aggregate["analysis_group"].map(
        group_order.set_index("analysis_group")["group_order"]
    )
    aggregate["group_code"] = aggregate["analysis_group"].map(
        group_codes
    )
    aggregate = aggregate.sort_values(
        [
            "group_order",
            "start_position_0based",
            "kmer",
        ]
    ).reset_index(drop=True)

    for frequency_field in (
        "organism_balanced_mean_frequency_percent",
        "pooled_frequency_percent",
    ):
        row_sums = aggregate.groupby(
            ["analysis_group", "start_position_0based"]
        )[frequency_field].sum()
        if not np.allclose(row_sums, 100.0, atol=1e-8):
            raise AssertionError(
                f"{frequency_field} does not sum to 100% for k={k}."
            )
    return aggregate


taxon_results = {
    k: aggregate_taxon_results(organism_level_results[k], k)
    for k in K_VALUES
}

for k, aggregate in taxon_results.items():
    aggregate.to_csv(
        OUTPUT_DIR / f"taxonwise_{k}mer_aggregate_tidy.csv",
        index=False,
        float_format="%.10f",
    )
    display(
        aggregate.query(
            "start_position_0based == 0 and "
            "kmer in ['CG', 'UA']"
        )[
            [
                "analysis_group",
                "kmer",
                "contributing_organisms",
                "pooled_eligible_unique_sequences",
                "organism_balanced_mean_frequency_percent",
                "pooled_frequency_percent",
            ]
        ]
    )


## Export group-specific wide tables


In [ ]:
def safe_group_folder(group_name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", group_name.lower()).strip("_")


for k, aggregate in taxon_results.items():
    motifs = ordered_kmers(k)
    for group_name in group_names:
        group_data = aggregate[
            aggregate["analysis_group"] == group_name
        ].copy()
        group_folder = (
            OUTPUT_DIR
            / f"{int(group_data['group_order'].iloc[0]):02d}_"
            f"{safe_group_folder(group_name)}"
        )
        group_folder.mkdir(parents=True, exist_ok=True)

        for value_field, file_label in (
            (
                "organism_balanced_mean_frequency_percent",
                "organism_balanced_frequency_percent",
            ),
            ("pooled_frequency_percent", "pooled_frequency_percent"),
            ("pooled_count", "pooled_counts"),
        ):
            wide = group_data.pivot(
                index="start_position_0based",
                columns="kmer",
                values=value_field,
            ).reindex(columns=motifs)
            wide.to_csv(
                group_folder / f"{k}mer_{file_label}.csv",
                float_format=(
                    "%d" if value_field == "pooled_count" else "%.10f"
                ),
            )

        support = (
            group_data[
                [
                    "start_position_0based",
                    "contributing_organisms",
                    "pooled_eligible_unique_sequences",
                ]
            ]
            .drop_duplicates()
            .sort_values("start_position_0based")
        )
        support.to_csv(
            group_folder / f"{k}mer_positional_support.csv",
            index=False,
        )


## Comparable editable SVG heatmaps


In [ ]:
mpl.rcParams.update(
    {
        "svg.fonttype": "none",
        "font.family": FONT_FAMILY,
        "axes.labelcolor": "#252525",
        "xtick.color": "#252525",
        "ytick.color": "#252525",
        "text.color": "#252525",
    }
)
sns.set_theme(style="white", context="paper")


common_vmax = {}
for k, aggregate in taxon_results.items():
    displayed = aggregate[
        aggregate["start_position_0based"].isin(DISPLAY_POSITIONS)
    ]
    common_vmax[k] = float(displayed[PRIMARY_FREQUENCY_FIELD].max())
print("Shared maximum colour scale by k:", common_vmax)


def plot_taxon_heatmap(
    group_name: str,
    k: int,
    value_field: str = PRIMARY_FREQUENCY_FIELD,
) -> Path:
    aggregate = taxon_results[k]
    motifs = ordered_kmers(k)
    group_data = aggregate[
        (aggregate["analysis_group"] == group_name)
        & (
            aggregate["start_position_0based"].isin(
                DISPLAY_POSITIONS
            )
        )
    ]
    plot_data = (
        group_data.pivot(
            index="start_position_0based",
            columns="kmer",
            values=value_field,
        )
        .reindex(index=DISPLAY_POSITIONS, columns=motifs)
    )
    if plot_data.isna().any().any():
        raise AssertionError(
            f"Missing displayed data for {group_name}, k={k}."
        )

    figure_width = 7.4 if k == 2 else 19.2
    figure_height = 7.5
    fig, ax = plt.subplots(
        figsize=(figure_width, figure_height),
        constrained_layout=True,
    )
    fig.patch.set_facecolor(FIGURE_BACKGROUND)
    ax.set_facecolor(FIGURE_BACKGROUND)

    sns.heatmap(
        plot_data,
        ax=ax,
        cmap=COLOR_MAP,
        vmin=0,
        vmax=common_vmax[k],
        linewidths=0.5,
        linecolor=GRID_COLOR,
        rasterized=False,
        cbar_kws={
            "label": "Mean frequency (%)",
            "shrink": 0.72,
            "pad": 0.025,
        },
    )
    ax.set_title(group_name, fontsize=13, fontweight="bold", pad=9)
    ax.set_xlabel(f"{k}-mer", fontsize=12, fontweight="bold")
    ax.set_ylabel(
        "K-mer start position (0-based)",
        fontsize=12,
        fontweight="bold",
    )
    ax.set_xticklabels(
        ax.get_xticklabels(),
        rotation=90 if k == 3 else 0,
        fontsize=7 if k == 3 else 9,
        fontweight="bold",
    )
    ax.set_yticklabels(
        [str(position) for position in DISPLAY_POSITIONS],
        rotation=0,
        fontsize=8,
    )
    ax.tick_params(axis="both", length=0)

    colorbar = ax.collections[0].colorbar
    colorbar.ax.set_ylabel(
        "Mean frequency (%)",
        fontsize=10,
        fontweight="bold",
        labelpad=10,
    )

    group_folder = (
        OUTPUT_DIR
        / f"{int(group_data['group_order'].iloc[0]):02d}_"
        f"{safe_group_folder(group_name)}"
    )
    label = (
        "organism_balanced"
        if value_field == PRIMARY_FREQUENCY_FIELD
        else "pooled"
    )
    output_path = (
        group_folder
        / f"{k}mer_{label}_positions_0_19_RdPu.svg"
    )
    fig.savefig(
        output_path,
        format="svg",
        bbox_inches="tight",
        facecolor=FIGURE_BACKGROUND,
        transparent=False,
        metadata={
            "Title": f"{group_name}: positional {k}-mer frequency",
            "Description": (
                "Exact sequences deduplicated within each organism; "
                f"{label} taxon aggregation; positions 0-19."
            ),
        },
    )
    plt.show()
    return output_path


In [ ]:
svg_paths = []
for group_name in group_names:
    for k in K_VALUES:
        svg_paths.append(
            plot_taxon_heatmap(
                group_name=group_name,
                k=k,
                value_field=PRIMARY_FREQUENCY_FIELD,
            )
        )
        if EXPORT_POOLED_HEATMAPS:
            svg_paths.append(
                plot_taxon_heatmap(
                    group_name=group_name,
                    k=k,
                    value_field="pooled_frequency_percent",
                )
            )

print(f"Saved {len(svg_paths)} SVG heatmaps.")


## Final support audit and provenance


In [ ]:
support_rows = []
for k, aggregate in taxon_results.items():
    support = (
        aggregate[
            [
                "group_order",
                "group_code",
                "analysis_group",
                "start_position_0based",
                "contributing_organisms",
                "pooled_eligible_unique_sequences",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            ["group_order", "start_position_0based"]
        )
    )
    support["k"] = k
    support_rows.append(support)
support_audit = pd.concat(support_rows, ignore_index=True)
support_audit.to_csv(
    OUTPUT_DIR / "taxonwise_positional_support_audit.csv",
    index=False,
)

provenance = {
    "analysis": "taxon-wise positional k-mer frequencies",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_fasta": str(FASTA_PATH.resolve()),
    "input_fasta_sha256": fasta_sha256,
    "group_manifest": str(GROUP_MANIFEST_PATH.resolve()),
    "selected_organisms": len(selected_prefixes),
    "selected_raw_records": len(selected_raw_records),
    "selected_unique_sequences_after_within_organism_deduplication": (
        observed_unique_total
    ),
    "selected_groups": EXPECTED_GROUP_SIZES,
    "k_values": list(K_VALUES),
    "overlapping_kmers": True,
    "displayed_start_positions": DISPLAY_POSITIONS,
    "primary_frequency": (
        "equal mean of organism-level positional frequencies"
    ),
    "sensitivity_frequency": (
        "pooled counts divided by pooled eligible sequences"
    ),
    "motif_order": {
        str(k): ordered_kmers(k) for k in K_VALUES
    },
    "shared_vmax_by_k": common_vmax,
    "software": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": mpl.__version__,
        "seaborn": sns.__version__,
    },
}
provenance_path = OUTPUT_DIR / "taxonwise_kmer_provenance.json"
provenance_path.write_text(
    json.dumps(provenance, indent=2),
    encoding="utf-8",
)

print("Analysis complete.")
print("Output directory:", OUTPUT_DIR.resolve())
print("Provenance:", provenance_path.resolve())
display(
    support_audit[
        support_audit["start_position_0based"].isin(
            [18, 19, 20, 21]
        )
    ].sort_values(
        ["k", "group_order", "start_position_0based"]
    )
)
